## 1. Setup

In [ ]:
import os, glob, random, math, json, re
import numpy as np
import nibabel as nib
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import pandas as pd

from skimage.metrics import peak_signal_noise_ratio as psnr_metric
from skimage.metrics import structural_similarity as ssim_metric
from sklearn.model_selection import KFold, StratifiedKFold, train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
CFG = dict(
    data_root="/kaggle/input/datasets/farahmo/longitudinal-mri-data/Longitudinal MR_data",
    modalities=["FLAIR", "T1", "T2"],
    modality_tags={"FLAIR": "flair", "T1": "t1", "T2": "t2"},
    slice_axis=2,
    img_size=128,
    min_brain_frac=0.02,

    train_frac=0.70,
    val_frac=0.15,

    batch_size=16,
    lr=3e-4,
    epochs=30,
    latent_dim=128,

    w_lesion_bce=1.0,
    w_lesion_dice=1.0,

    use_lr_schedule=True,
    lr_min_frac=0.01,
    early_stop_metric="dice",
    early_stopping_patience=6,

    augment=True,
    aug_flip_prob=0.5,
    aug_rot90_prob=0.5,
    aug_intensity_jitter=0.05,

    use_pos_weighting=True,
    use_focal_loss=False,
    focal_gamma=2.0,

    allow_missing_modality=False,

    kfold_n_splits=5,

    num_workers=2,
    use_amp=True,

    demographics_csv="/kaggle/input/datasets/farahmo/longitudinal-mri-data/Longitudinal MR_data/long-MR-MS_demographics (1).csv",
    stratify_by="ms_type",
    seed=SEED,
)

CFG

In [ ]:
# Identical to the original notebook: same 5 seeds, same protocol toggles.
N_SEEDS = 5
SEEDS = [42, 43, 44, 45, 46]
RUN_MULTISEED = True
RUN_KFOLD = True

CFG["kfold_n_splits"] = 5
CFG["early_stop_metric"] = "dice"

assert len(SEEDS) == N_SEEDS
print(f"N_SEEDS={N_SEEDS}, SEEDS={SEEDS}")
print(f"RUN_MULTISEED={RUN_MULTISEED}  RUN_KFOLD={RUN_KFOLD}  kfold_n_splits={CFG['kfold_n_splits']}")

In [ ]:
_PATIENT_DIR_RE = re.compile(r"^patient\s*\d+$", re.IGNORECASE)


def list_patients(root):
    found = []
    for dirpath, dirnames, filenames in os.walk(root):
        base = os.path.basename(dirpath.rstrip("/\\\\"))
        if not _PATIENT_DIR_RE.match(base):
            continue
        has_gt = any("gt" in fn.lower() and (fn.lower().endswith(".nii") or fn.lower().endswith(".nii.gz"))
                      for fn in filenames)
        if has_gt:
            found.append(dirpath)
    return sorted(set(found))


def find_one(patient_dir, *substrings):
    candidates = []
    for f in os.listdir(patient_dir):
        full = os.path.join(patient_dir, f)
        if os.path.isdir(full):
            continue
        low = f.lower()
        if not (low.endswith(".nii") or low.endswith(".nii.gz")):
            continue
        if all(s.lower() in low for s in substrings):
            candidates.append(full)
    if not candidates:
        return None
    candidates.sort(key=len)
    return candidates[0]


def find_patient_files(pdir, modality_tags, allow_missing_modality=False):
    brainmask_path = find_one(pdir, "brainmask")
    gt_path = find_one(pdir, "gt")
    if brainmask_path is None:
        raise FileNotFoundError(f"no *brainmask*.nii[.gz] file in {pdir}")
    if gt_path is None:
        raise FileNotFoundError(f"no *gt*.nii[.gz] file in {pdir}")

    study1_paths, study2_paths = {}, {}
    for m, tag in modality_tags.items():
        p1 = find_one(pdir, "study1", tag)
        p2 = find_one(pdir, "study2", tag)
        if p1 is None:
            if not allow_missing_modality:
                raise FileNotFoundError(f"no study1/{m} ('{tag}') file in {pdir}")
            print(f"  [warn] {os.path.basename(pdir)}: missing study1/{m} -> will zero-fill this channel")
        if p2 is None:
            if not allow_missing_modality:
                raise FileNotFoundError(f"no study2/{m} ('{tag}') file in {pdir}")
            print(f"  [warn] {os.path.basename(pdir)}: missing study2/{m} -> will zero-fill this channel")
        study1_paths[m] = p1
        study2_paths[m] = p2

    return brainmask_path, gt_path, study1_paths, study2_paths


def load_nifti(path):
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    return nib.load(path).get_fdata(dtype=np.float32)


def normalize_volume(vol, mask=None, eps=1e-6):
    """Robust (1st-99th percentile) intensity normalisation within the brain mask."""
    vals = vol[mask > 0] if mask is not None else vol.flatten()
    if vals.size == 0:
        return np.zeros_like(vol)
    p1, p99 = np.percentile(vals, [1, 99])
    vol = np.clip(vol, p1, p99)
    vol = (vol - p1) / (p99 - p1 + eps)
    if mask is not None:
        vol = vol * (mask > 0)
    return vol.astype(np.float32)


def resize2d(arr, size):
    t = torch.from_numpy(arr)[None, None].float()
    t = F.interpolate(t, size=(size, size), mode="bilinear", align_corners=False)
    return t[0, 0].numpy()

In [ ]:
def augment_2d(x1, x2, lesion, brain, cfg):
    if torch.rand(()).item() < cfg["aug_flip_prob"]:
        x1 = torch.flip(x1, dims=[-1]); x2 = torch.flip(x2, dims=[-1])
        lesion = torch.flip(lesion, dims=[-1]); brain = torch.flip(brain, dims=[-1])
    if torch.rand(()).item() < cfg["aug_flip_prob"]:
        x1 = torch.flip(x1, dims=[-2]); x2 = torch.flip(x2, dims=[-2])
        lesion = torch.flip(lesion, dims=[-2]); brain = torch.flip(brain, dims=[-2])
    if torch.rand(()).item() < cfg["aug_rot90_prob"]:
        k = int(torch.randint(1, 4, (1,)).item())
        x1 = torch.rot90(x1, k, dims=[-2, -1]); x2 = torch.rot90(x2, k, dims=[-2, -1])
        lesion = torch.rot90(lesion, k, dims=[-2, -1]); brain = torch.rot90(brain, k, dims=[-2, -1])

    jitter = cfg["aug_intensity_jitter"]
    if jitter > 0:
        C = x1.shape[0]
        gain1 = 1.0 + (torch.rand(C, 1, 1) * 2 - 1) * jitter
        off1 = (torch.rand(C, 1, 1) * 2 - 1) * jitter
        x1 = (x1 * gain1 + off1).clamp(0, 1)
        gain2 = 1.0 + (torch.rand(C, 1, 1) * 2 - 1) * jitter
        off2 = (torch.rand(C, 1, 1) * 2 - 1) * jitter
        x2 = (x2 * gain2 + off2).clamp(0, 1)

    return x1.contiguous(), x2.contiguous(), lesion.contiguous(), brain.contiguous()


class SliceCache:
    """Loads NIfTI volumes for given patient directories and precomputes the
    normalized + resized 2D slice arrays ONCE, at construction time."""

    def __init__(self, patient_dirs, cfg, demographics_df=None):
        self.cfg = cfg
        self.slice_axis = cfg["slice_axis"]
        self.pid_samples = {}
        self.samples = []

        for pdir in patient_dirs:
            pid = os.path.basename(pdir)
            try:
                bm_path, gt_path, s1_paths, s2_paths = find_patient_files(
                    pdir, cfg["modality_tags"], allow_missing_modality=cfg["allow_missing_modality"]
                )
                brainmask = load_nifti(bm_path)
                gt = load_nifti(gt_path)

                study1 = {m: (load_nifti(p) if p is not None else np.zeros_like(brainmask))
                          for m, p in s1_paths.items()}
                study2 = {m: (load_nifti(p) if p is not None else np.zeros_like(brainmask))
                          for m, p in s2_paths.items()}
            except FileNotFoundError as e:
                print(f"[skip] {pid}: {e}")
                continue

            dt_days = get_days_between_studies(demographics_df, pid)
            dt_val = np.float32(dt_days / 365.25)

            n_slices = brainmask.shape[self.slice_axis]
            recs = []
            for s in range(n_slices):
                bm_slice = np.take(brainmask, s, axis=self.slice_axis)
                if bm_slice.mean() < cfg["min_brain_frac"]:
                    continue
                gt_slice = np.take(gt, s, axis=self.slice_axis)

                x1_chs, x2_chs = [], []
                for m in cfg["modalities"]:
                    sl1 = normalize_volume(np.take(study1[m], s, axis=self.slice_axis), bm_slice)
                    sl2 = normalize_volume(np.take(study2[m], s, axis=self.slice_axis), bm_slice)
                    x1_chs.append(resize2d(sl1, cfg["img_size"]))
                    x2_chs.append(resize2d(sl2, cfg["img_size"]))

                x1 = np.stack(x1_chs, 0).astype(np.float32)
                x2 = np.stack(x2_chs, 0).astype(np.float32)
                lesion = resize2d((gt_slice > 0).astype(np.float32), cfg["img_size"])[None].astype(np.float32)
                brain = resize2d((bm_slice > 0).astype(np.float32), cfg["img_size"])[None].astype(np.float32)

                recs.append(dict(x1=x1, x2=x2, lesion=lesion, brain=brain, slice_idx=s, dt=dt_val))

            if recs:
                self.pid_samples[pid] = recs
                self.samples.extend((pid, i) for i in range(len(recs)))

        n_patients = len(self.pid_samples)
        print(f"[SliceCache] loaded + precomputed {n_patients} patient(s), "
              f"{len(self.samples)} usable slice(s).")


class MSLongitudinalSliceDataset(Dataset):
    """Thin, augmentation-only view over a SliceCache."""

    def __init__(self, patient_dirs, cfg, train=False, demographics_df=None, cache=None):
        self.cfg = cfg
        self.train = train

        if cache is not None:
            self.cache = cache
            wanted = set(patient_dirs)
        else:
            self.cache = SliceCache(patient_dirs, cfg, demographics_df=demographics_df)
            wanted = set(self.cache.pid_samples.keys())

        self.samples = [(pid, i) for (pid, i) in self.cache.samples if pid in wanted]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        pid, i = self.samples[idx]
        rec = self.cache.pid_samples[pid][i]

        x1 = torch.from_numpy(rec["x1"]).clone()
        x2 = torch.from_numpy(rec["x2"]).clone()
        lesion = torch.from_numpy(rec["lesion"]).clone()
        brain = torch.from_numpy(rec["brain"]).clone()

        if self.train and self.cfg.get("augment", False):
            x1, x2, lesion, brain = augment_2d(x1, x2, lesion, brain, self.cfg)

        dt = torch.tensor(float(rec["dt"]), dtype=torch.float32)

        return dict(x1=x1, x2=x2, lesion=lesion, brainmask=brain, pid=pid,
                    slice_idx=rec["slice_idx"], dt=dt)

In [ ]:
def get_days_between_studies(demographics_df, pid, default=365.25):
    if demographics_df is None:
        return default
    matches = [c for c in demographics_df.columns if c.lower() == "days_between_studies"]
    if not matches:
        return default
    val = demographics_df[matches[0]].get(pid, None)
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return default
    return float(val)


def load_demographics(csv_path):
    if csv_path is None:
        print("[demographics] CFG['demographics_csv'] is None -- skipping.")
        return None
    if not os.path.exists(csv_path):
        print(f"[demographics] file not found at {csv_path} -- skipping "
              f"(patient split will fall back to plain random).")
        return None

    df = pd.read_csv(csv_path)
    df.columns = [c.strip() for c in df.columns]

    id_col = None
    for candidate in ["patient_id", "patient_ids", "patientid", "id", "patient"]:
        matches = [c for c in df.columns if c.lower() == candidate]
        if matches:
            id_col = matches[0]
            break
    if id_col is None:
        print(f"[demographics] could not find a patient-id column in {csv_path}; "
              f"columns found: {list(df.columns)} -- skipping.")
        return None

    extracted = df[id_col].astype(str).str.extract(r"(\d+)")[0]
    valid = extracted.notna()
    if not valid.all():
        print(f"[demographics] {(~valid).sum()} row(s) had no numeric patient id in "
              f"column '{id_col}' and will be dropped.")
    df = df.loc[valid].copy()
    df["pid"] = extracted.loc[valid].astype(int).apply(lambda x: f"patient{x}")
    df = df.set_index("pid")
    return df


demographics_df = load_demographics(CFG["demographics_csv"])
if demographics_df is not None:
    print(f"Loaded demographics for {len(demographics_df)} patients.")

In [ ]:
patient_dirs = list_patients(CFG["data_root"])
print(f"Found {len(patient_dirs)} patients under {CFG['data_root']}")
assert len(patient_dirs) > 0, "No patient folders found -- check CFG['data_root']."


def stratified_patient_split(patient_dirs, cfg, demographics_df):
    pids = [os.path.basename(p) for p in patient_dirs]
    strat_col = cfg.get("stratify_by")

    can_stratify = (
        demographics_df is not None
        and strat_col is not None
        and any(c.lower() == strat_col.lower() for c in demographics_df.columns)
    )

    if can_stratify:
        actual_col = [c for c in demographics_df.columns if c.lower() == strat_col.lower()][0]
        labels = pd.Series(pids).map(lambda pid: demographics_df[actual_col].get(pid, None))
        label_counts = labels.value_counts(dropna=False)
        min_class_size = label_counts.min() if len(label_counts) > 0 else 0
        has_missing = labels.isna().any()

        if has_missing or len(label_counts) < 2 or min_class_size < 2:
            print(f"[split] stratification by '{actual_col}' not viable -- falling back to plain random split.")
            can_stratify = False

    if can_stratify:
        actual_col = [c for c in demographics_df.columns if c.lower() == strat_col.lower()][0]
        labels = [demographics_df[actual_col].get(pid) for pid in pids]
        test_frac = 1.0 - cfg["train_frac"] - cfg["val_frac"]
        try:
            train_pids, temp_pids, train_labels, temp_labels = train_test_split(
                pids, labels, test_size=(cfg["val_frac"] + test_frac),
                stratify=labels, random_state=cfg["seed"]
            )
            rel_val_frac = cfg["val_frac"] / (cfg["val_frac"] + test_frac)
            val_pids, test_pids, _, _ = train_test_split(
                temp_pids, temp_labels, test_size=(1 - rel_val_frac),
                stratify=temp_labels, random_state=cfg["seed"]
            )
            print(f"[split] stratified by '{actual_col}'.")
        except ValueError as e:
            print(f"[split] stratified split failed ({e}) -- falling back to plain random split.")
            can_stratify = False

    if not can_stratify:
        rng = random.Random(cfg["seed"])
        shuffled = pids[:]
        rng.shuffle(shuffled)
        n = len(shuffled)
        n_train = max(1, int(cfg["train_frac"] * n))
        n_val = max(1, int(cfg["val_frac"] * n))
        train_pids = shuffled[:n_train]
        val_pids = shuffled[n_train:n_train + n_val]
        test_pids = shuffled[n_train + n_val:]
        print("[split] plain random split (no demographics-based stratification).")

    pid_to_dir = {os.path.basename(p): p for p in patient_dirs}
    return ([pid_to_dir[p] for p in train_pids],
            [pid_to_dir[p] for p in val_pids],
            [pid_to_dir[p] for p in test_pids])


train_p, val_p, test_p = stratified_patient_split(patient_dirs, CFG, demographics_df)
print(f"patients -> train {len(train_p)} / val {len(val_p)} / test {len(test_p)}")

train_ds = MSLongitudinalSliceDataset(train_p, CFG, train=True, demographics_df=demographics_df)
val_ds = MSLongitudinalSliceDataset(val_p, CFG, train=False, demographics_df=demographics_df)
test_ds = MSLongitudinalSliceDataset(test_p, CFG, train=False, demographics_df=demographics_df)
print(f"slices -> train {len(train_ds)} / val {len(val_ds)} / test {len(test_ds)}")

_NW = CFG.get("num_workers", 0)
train_dl = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True, num_workers=_NW,
                       pin_memory=True, persistent_workers=(_NW > 0), drop_last=True,
                       generator=torch.Generator().manual_seed(CFG["seed"]))
val_dl = DataLoader(val_ds, batch_size=CFG["batch_size"], shuffle=False, num_workers=_NW,
                     pin_memory=True, persistent_workers=(_NW > 0))
test_dl = DataLoader(test_ds, batch_size=CFG["batch_size"], shuffle=False, num_workers=_NW,
                      pin_memory=True, persistent_workers=(_NW > 0))

In [ ]:
def conv_block(cin, cout, k=3, s=1, p=1):
    return nn.Sequential(
        nn.Conv2d(cin, cout, k, s, p),
        nn.BatchNorm2d(cout),
        nn.ReLU(inplace=True),
    )


class Encoder(nn.Module):
    def __init__(self, in_ch, base=32, latent_ch=128):
        super().__init__()
        self.enc1 = nn.Sequential(conv_block(in_ch, base), conv_block(base, base))
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = nn.Sequential(conv_block(base, base * 2), conv_block(base * 2, base * 2))
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = nn.Sequential(conv_block(base * 2, base * 4), conv_block(base * 4, base * 4))
        self.pool3 = nn.MaxPool2d(2)
        self.bottleneck = conv_block(base * 4, latent_ch)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        z = self.bottleneck(self.pool3(e3))
        return z, (e1, e2, e3)


class Decoder(nn.Module):
    def __init__(self, latent_ch=128, base=32, out_ch=3, use_skips=True):
        super().__init__()
        self.use_skips = use_skips
        mult = 2 if use_skips else 1
        self.up3 = nn.ConvTranspose2d(latent_ch, base * 4, 2, 2)
        self.dec3 = nn.Sequential(conv_block(base * 4 * mult, base * 4), conv_block(base * 4, base * 4))
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, 2)
        self.dec2 = nn.Sequential(conv_block(base * 2 * mult, base * 2), conv_block(base * 2, base * 2))
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, 2)
        self.dec1 = nn.Sequential(conv_block(base * mult, base), conv_block(base, base))
        self.out = nn.Conv2d(base, out_ch, 1)

    def forward(self, z, skips=None):
        d3 = self.up3(z)
        if self.use_skips and skips is not None:
            d3 = torch.cat([d3, skips[2]], dim=1)
        d3 = self.dec3(d3)

        d2 = self.up2(d3)
        if self.use_skips and skips is not None:
            d2 = torch.cat([d2, skips[1]], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        if self.use_skips and skips is not None:
            d1 = torch.cat([d1, skips[0]], dim=1)
        d1 = self.dec1(d1)

        return self.out(d1)


class DirectUNetBaseline(nn.Module):
    """SOTA-style baseline: plain encoder-decoder image-to-image forecasting,
    no explicit latent-dynamics module (cf. Farki et al. 2025, arXiv:2511.02558).
    Decodes directly from the encoder's bottleneck -- no residual
    "z2_hat = z1 + f(z1)" step -- so there is no separate temporal-dynamics
    claim to compare against WorldModelBase / SCL-Latent's explicit dynamics
    module."""

    def __init__(self, in_ch, out_ch, latent_ch=128, base=32):
        super().__init__()
        self.encoder = Encoder(in_ch, base=base, latent_ch=latent_ch)
        self.decoder = Decoder(latent_ch, base=base, out_ch=out_ch, use_skips=True)
        self.lesion_head = nn.Sequential(
            nn.Conv2d(latent_ch, 32, 3, 1, 1), nn.ReLU(inplace=True), nn.Conv2d(32, 1, 1)
        )

    def forward(self, x1):
        z1, skips = self.encoder(x1)
        x2_hat = self.decoder(z1, skips)
        lesion_logits = F.interpolate(self.lesion_head(z1), size=x1.shape[-2:],
                                       mode="bilinear", align_corners=False)
        return dict(x2_hat=x2_hat, lesion_logits=lesion_logits, z1=z1)


def count_params(m):
    return sum(p.numel() for p in m.parameters())

In [ ]:
def dice_loss(logits, target, eps=1e-6):
    probs = torch.sigmoid(logits).flatten(1)
    target = target.flatten(1)
    inter = (probs * target).sum(1)
    union = probs.sum(1) + target.sum(1)
    dice = (2 * inter + eps) / (union + eps)
    return 1 - dice.mean()


def compute_pos_weight(dl, max_batches=20, cap=100.0):
    """Estimates BCEWithLogitsLoss pos_weight = (#negative voxels)/(#positive voxels)."""
    pos, neg = 0.0, 0.0
    for i, batch in enumerate(dl):
        if i >= max_batches:
            break
        y = batch["lesion"]
        pos += y.sum().item()
        neg += (1.0 - y).sum().item()
    if pos == 0:
        print("[warn] compute_pos_weight: no positive voxels found -- falling back to pos_weight=1.0")
        return 1.0
    return float(min(neg / pos, cap))


def direct_loss(out, x2, lesion, cfg, pos_weight=None):
    recon = F.l1_loss(out["x2_hat"], x2)

    if cfg.get("use_pos_weighting", False) and pos_weight is not None:
        pw = torch.as_tensor(pos_weight, dtype=torch.float32, device=out["lesion_logits"].device)
        lesion_bce = F.binary_cross_entropy_with_logits(out["lesion_logits"], lesion, pos_weight=pw)
    else:
        lesion_bce = F.binary_cross_entropy_with_logits(out["lesion_logits"], lesion)

    lesion_dc = dice_loss(out["lesion_logits"], lesion)

    total = recon + cfg["w_lesion_bce"] * lesion_bce + cfg["w_lesion_dice"] * lesion_dc
    logs = dict(recon=recon.item(), lesion_bce=lesion_bce.item(), lesion_dice=lesion_dc.item(), total=total.item())
    return total, logs


def run_epoch(model, dl, cfg, optimizer=None, pos_weight=None, scaler=None, use_amp=False):
    train_mode = optimizer is not None
    model.train() if train_mode else model.eval()

    agg, n = {}, 0
    for batch in dl:
        x1 = batch["x1"].to(device, non_blocking=True)
        x2 = batch["x2"].to(device, non_blocking=True)
        lesion = batch["lesion"].to(device, non_blocking=True)

        with torch.set_grad_enabled(train_mode):
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                out = model(x1)
                loss, logs = direct_loss(out, x2, lesion, cfg, pos_weight=pos_weight)
            if train_mode:
                optimizer.zero_grad(set_to_none=True)
                if use_amp:
                    scaler.scale(loss).backward()
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()

        bs = x1.size(0)
        n += bs
        for k, v in logs.items():
            agg[k] = agg.get(k, 0.0) + v * bs

    return {k: v / n for k, v in agg.items()}


def train_model(model, name, train_dl, val_dl, cfg, epochs=None, pos_weight=None, seed=None):
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)

    epochs = epochs or cfg["epochs"]
    opt = torch.optim.Adam(model.parameters(), lr=cfg["lr"], weight_decay=1e-4)

    use_amp = bool(cfg.get("use_amp", False)) and device.type == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    scheduler = None
    if cfg.get("use_lr_schedule", False):
        eta_min = cfg["lr"] * cfg.get("lr_min_frac", 0.01)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=eta_min)

    metric_key = cfg.get("early_stop_metric", "dice")
    patience = cfg.get("early_stopping_patience", None)

    def score_of(va):
        return va["lesion_dice"] if metric_key == "dice" else va["total"]

    history = {"train": [], "val": []}
    best_score, best_state, best_epoch = float("inf"), None, 0
    epochs_since_improve = 0

    for ep in range(1, epochs + 1):
        tr = run_epoch(model, train_dl, cfg, optimizer=opt, pos_weight=pos_weight, scaler=scaler, use_amp=use_amp)
        va = run_epoch(model, val_dl, cfg, optimizer=None, pos_weight=pos_weight, scaler=scaler, use_amp=use_amp)
        history["train"].append(tr)
        history["val"].append(va)

        score = score_of(va)
        improved = score < best_score - 1e-6
        if improved:
            best_score, best_epoch = score, ep
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            epochs_since_improve = 0
        else:
            epochs_since_improve += 1

        lr_now = opt.param_groups[0]["lr"]
        marker = " *" if improved else ""
        print(f"[{name}] epoch {ep:02d}/{epochs}  "
              f"train_loss={tr['total']:.4f}  val_loss={va['total']:.4f}  "
              f"val_dice={1 - va['lesion_dice']:.4f}  lr={lr_now:.2e}{marker}")

        if scheduler is not None:
            scheduler.step()

        if patience is not None and epochs_since_improve >= patience:
            print(f"[{name}] early stopping at epoch {ep} (no improvement for {patience} epochs; best was epoch {best_epoch})")
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history

In [ ]:
@torch.no_grad()
def evaluate_model(model, dl, name):
    model.eval()
    per_patient = {}

    for batch in dl:
        x1 = batch["x1"].to(device)
        x2 = batch["x2"].to(device)
        lesion = batch["lesion"].to(device)
        pids = batch["pid"]
        out = model(x1)

        x2_hat = out["x2_hat"].clamp(0, 1).cpu().numpy()
        x2_np = x2.cpu().numpy()

        probs = torch.sigmoid(out["lesion_logits"])
        pred = (probs > 0.5).float()
        inter = (pred * lesion).sum(dim=tuple(range(1, lesion.dim())))
        union = pred.sum(dim=tuple(range(1, lesion.dim()))) + lesion.sum(dim=tuple(range(1, lesion.dim())))
        dice_per_sample = ((2 * inter + 1e-6) / (union + 1e-6)).cpu().numpy()

        for b, pid in enumerate(pids):
            entry = per_patient.setdefault(pid, dict(psnr=[], ssim=[], dice=[]))
            for c in range(x2_np.shape[1]):
                entry["psnr"].append(psnr_metric(x2_np[b, c], x2_hat[b, c], data_range=1.0))
                entry["ssim"].append(ssim_metric(x2_np[b, c], x2_hat[b, c], data_range=1.0))
            entry["dice"].append(float(dice_per_sample[b]))

    patient_psnr = [float(np.mean(v["psnr"])) for v in per_patient.values()]
    patient_ssim = [float(np.mean(v["ssim"])) for v in per_patient.values()]
    patient_dice = [float(np.mean(v["dice"])) for v in per_patient.values()]

    return dict(
        model=name,
        n_patients=len(per_patient),
        psnr_mean=float(np.mean(patient_psnr)), psnr_std=float(np.std(patient_psnr)),
        ssim_mean=float(np.mean(patient_ssim)), ssim_std=float(np.std(patient_ssim)),
        dice_mean=float(np.mean(patient_dice)), dice_std=float(np.std(patient_dice)),
    )

In [ ]:
in_ch = out_ch = len(CFG["modalities"])

model_direct = DirectUNetBaseline(in_ch, out_ch, latent_ch=CFG["latent_dim"]).to(device)
print(f"DirectUNetBaseline: {count_params(model_direct):,} params")

pos_weight_2d = compute_pos_weight(train_dl) if CFG.get("use_pos_weighting", False) else None
if pos_weight_2d is not None:
    print(f"Estimated BCE pos_weight from training data: {pos_weight_2d:.2f}")

print("\nTraining DirectUNetBaseline (primary split)...")
model_direct, hist_direct = train_model(model_direct, "DirectUNetBaseline", train_dl, val_dl, CFG,
                                         pos_weight=pos_weight_2d)

res_direct_primary = evaluate_model(model_direct, test_dl, "DirectUNetBaseline")
print(f"\nPrimary-split result (n={res_direct_primary['n_patients']} test patients):")
print(pd.DataFrame([res_direct_primary]).set_index("model")[
    ["n_patients", "psnr_mean", "psnr_std", "ssim_mean", "ssim_std", "dice_mean", "dice_std"]])

In [ ]:
def build_seeded_train_dl(seed):
    nw = CFG.get("num_workers", 0)
    return DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True, num_workers=nw,
                       pin_memory=True, persistent_workers=False, drop_last=True,
                       generator=torch.Generator().manual_seed(seed))


multiseed_records = []

if RUN_MULTISEED:
    for s in SEEDS:
        print(f"\n========== seed {s} ==========")
        train_dl_s = build_seeded_train_dl(s)

        m_direct_s = DirectUNetBaseline(in_ch, out_ch, latent_ch=CFG["latent_dim"]).to(device)
        pw_s = compute_pos_weight(train_dl_s) if CFG.get("use_pos_weighting", False) else None

        m_direct_s, _ = train_model(m_direct_s, f"seed{s}-DirectUNet", train_dl_s, val_dl, CFG,
                                     pos_weight=pw_s, seed=s)

        r_s = evaluate_model(m_direct_s, test_dl, "DirectUNetBaseline")
        r_s["seed"] = s
        multiseed_records.append(r_s)

    multiseed_df = pd.DataFrame(multiseed_records)
    print("\nPer-seed patient-weighted results:")
    print(multiseed_df[["model", "seed", "psnr_mean", "ssim_mean", "dice_mean"]])

    multiseed_summary = multiseed_df.groupby("model")[["psnr_mean", "ssim_mean", "dice_mean"]].agg(["mean", "std"])
    print(f"\nAcross-seed summary (n={N_SEEDS} seeds, mean +/- std, patient-level PRIMARY metric):")
    print(multiseed_summary)

    os.makedirs("outputs", exist_ok=True)
    multiseed_df.to_csv("outputs/direct_unet_multiseed.csv", index=False)
else:
    multiseed_df, multiseed_summary = None, None
    print("RUN_MULTISEED is False -- skipping.")

In [ ]:
def run_kfold_cv_direct(patient_dirs, cfg, k=5, epochs=None, val_frac_within_fold=0.15,
                         demographics_df=None, seeds=None,
                         checkpoint_path="outputs/kfold_checkpoint_direct_unet.csv"):
    """Same design as run_kfold_cv / run_kfold_cv_curvature in the original notebook,
    but trains and evaluates DirectUNetBaseline. Uses the identical fold splits
    (same random_state) so this checkpoint is directly comparable row-for-row
    on (fold, seed) with the original notebook's kfold_checkpoint_2d.csv and
    kfold_checkpoint_curvature_2d.csv."""
    seeds = list(seeds) if seeds is not None else [cfg["seed"]]
    nw = cfg.get("num_workers", 0)

    os.makedirs(os.path.dirname(checkpoint_path) or ".", exist_ok=True)
    done_keys = set()
    prev_rows = []
    if os.path.exists(checkpoint_path):
        prev_df = pd.read_csv(checkpoint_path)
        prev_rows = prev_df.to_dict("records")
        done_keys = set(zip(prev_df["fold"], prev_df["seed"], prev_df["model"]))
        print(f"[kfold-direct] resuming from checkpoint: {len(prev_df)} run(s) already completed in {checkpoint_path}")

    pids = [os.path.basename(p) for p in patient_dirs]
    pid_to_dir = {os.path.basename(p): p for p in patient_dirs}
    strat_col_name = cfg.get("stratify_by")

    can_stratify = False
    fold_labels = None
    if demographics_df is not None and strat_col_name is not None:
        matches = [c for c in demographics_df.columns if c.lower() == strat_col_name.lower()]
        if matches:
            actual_col = matches[0]
            fold_labels = [demographics_df[actual_col].get(pid) for pid in pids]
            has_missing = any(l is None or (isinstance(l, float) and np.isnan(l)) for l in fold_labels)
            label_counts = pd.Series(fold_labels).value_counts(dropna=False)
            min_class_size = label_counts.min() if len(label_counts) > 0 else 0
            if not has_missing and min_class_size >= k:
                can_stratify = True

    # Same random_state as the original notebook's run_kfold_cv (cfg["seed"]) =>
    # identical fold membership, so results are directly comparable.
    if can_stratify:
        kf = StratifiedKFold(n_splits=k, shuffle=True, random_state=cfg["seed"])
        split_iter = list(kf.split(pids, fold_labels))
        print(f"[kfold-direct] using StratifiedKFold on '{strat_col_name}' (k={k}).")
    else:
        kf = KFold(n_splits=k, shuffle=True, random_state=cfg["seed"])
        split_iter = list(kf.split(pids))
        print(f"[kfold-direct] using plain KFold (k={k}, no stratification).")

    print(f"[kfold-direct] training under {len(seeds)} seed(s) per fold: {seeds}  "
          f"({k} folds x {len(seeds)} seeds = {k * len(seeds)} runs for DirectUNetBaseline)")

    fold_results = list(prev_rows)

    for fold_i, (train_idx, test_idx) in enumerate(split_iter):
        train_val_pids = [pids[i] for i in train_idx]
        test_pids_fold = [pids[i] for i in test_idx]
        train_val_dirs = [pid_to_dir[p] for p in train_val_pids]
        test_dirs = [pid_to_dir[p] for p in test_pids_fold]

        all_done_for_fold = all((fold_i + 1, s, "DirectUNetBaseline") in done_keys for s in seeds)
        if all_done_for_fold:
            print(f"[kfold-direct] fold {fold_i + 1}/{k}: all seeds already in checkpoint -- skipping load.")
            continue

        print(f"\n[kfold-direct] fold {fold_i + 1}/{k}: loading {len(train_val_dirs)} train/val "
              f"+ {len(test_dirs)} test patient(s) into cache...")
        train_val_cache = SliceCache(train_val_dirs, cfg, demographics_df=demographics_df)
        test_cache = SliceCache(test_dirs, cfg, demographics_df=demographics_df)

        test_ds_f = MSLongitudinalSliceDataset(test_pids_fold, cfg, train=False, cache=test_cache)
        if len(test_ds_f) == 0:
            print(f"  [skip fold {fold_i + 1}]: empty test split")
            continue
        test_dl_f = DataLoader(test_ds_f, batch_size=cfg["batch_size"], shuffle=False,
                                num_workers=nw, pin_memory=True, persistent_workers=(nw > 0))

        n_val = max(1, int(len(train_val_pids) * val_frac_within_fold))

        for seed_i, s in enumerate(seeds):
            if (fold_i + 1, s, "DirectUNetBaseline") in done_keys:
                print(f"  [kfold-direct] fold {fold_i + 1} seed {s}: already in checkpoint -- skipping")
                continue

            print(f"\n===== Fold {fold_i + 1}/{k}  |  seed {s} ({seed_i + 1}/{len(seeds)}) =====")

            rng_fold = random.Random(s + fold_i)
            shuffled = train_val_pids[:]
            rng_fold.shuffle(shuffled)
            val_pids_fold = shuffled[:n_val]
            train_pids_fold = shuffled[n_val:]

            train_ds_f = MSLongitudinalSliceDataset(train_pids_fold, cfg, train=True, cache=train_val_cache)
            val_ds_f = MSLongitudinalSliceDataset(val_pids_fold, cfg, train=False, cache=train_val_cache)

            if len(train_ds_f) == 0 or len(val_ds_f) == 0:
                print(f"  [skip fold {fold_i + 1} seed {s}]: an empty split (train={len(train_ds_f)}, val={len(val_ds_f)})")
                continue

            train_dl_f = DataLoader(train_ds_f, batch_size=cfg["batch_size"], shuffle=True,
                                     num_workers=nw, pin_memory=True, persistent_workers=(nw > 0),
                                     drop_last=True, generator=torch.Generator().manual_seed(s + fold_i))
            val_dl_f = DataLoader(val_ds_f, batch_size=cfg["batch_size"], shuffle=False,
                                   num_workers=nw, pin_memory=True, persistent_workers=(nw > 0))

            in_ch_f = out_ch_f = len(cfg["modalities"])
            m_direct = DirectUNetBaseline(in_ch_f, out_ch_f, latent_ch=cfg["latent_dim"]).to(device)

            pw = compute_pos_weight(train_dl_f) if cfg.get("use_pos_weighting", False) else None

            m_direct, _ = train_model(m_direct, f"fold{fold_i + 1}-seed{s}-DirectUNet", train_dl_f, val_dl_f, cfg,
                                       epochs=epochs or cfg["epochs"], pos_weight=pw, seed=s)

            rd = evaluate_model(m_direct, test_dl_f, "DirectUNetBaseline")
            rd["fold"] = fold_i + 1; rd["seed"] = s
            fold_results.append(rd)

            pd.DataFrame([rd]).to_csv(
                checkpoint_path, mode="a", header=not os.path.exists(checkpoint_path), index=False
            )
            done_keys.add((fold_i + 1, s, "DirectUNetBaseline"))
            print(f"  [checkpoint] saved fold {fold_i + 1} seed {s} result -> {checkpoint_path}")

    fold_df = pd.DataFrame(fold_results)
    summary = fold_df.groupby("model")[["psnr_mean", "ssim_mean", "dice_mean"]].agg(["mean", "std"])
    return fold_df, summary


if RUN_KFOLD:
    fold_df_direct, kfold_summary_direct = run_kfold_cv_direct(
        patient_dirs, CFG, k=CFG["kfold_n_splits"], demographics_df=demographics_df, seeds=SEEDS
    )
    print("\nPer-fold-per-seed results:")
    print(fold_df_direct[["model", "fold", "seed", "psnr_mean", "ssim_mean", "dice_mean"]])
    print(f"\nAcross-fold x seed summary (patient-level, mean +/- std, "
          f"k={CFG['kfold_n_splits']} folds x {len(SEEDS)} seeds = {CFG['kfold_n_splits'] * len(SEEDS)} runs):")
    print(kfold_summary_direct)
else:
    fold_df_direct, kfold_summary_direct = None, None
    print("RUN_KFOLD is False -- skipping.")

In [ ]:
REFERENCE_RESULTS = {
    "multiseed": {   # Table 1: primary split, S=5 seeds, mean +/- std
        "Baseline":   dict(psnr=(26.95, 0.19), ssim=(0.864, 0.039), dice=(0.274, 0.025)),
        "SCL-Latent": dict(psnr=(27.13, 0.17), ssim=(0.831, 0.041), dice=(0.255, 0.037)),
    },
    "kfold": {   # Table 2 / 4: k=5 folds x S=5 seeds = 25 runs, mean +/- std
        "Baseline":          dict(psnr=(22.75, 3.16), ssim=(0.639, 0.169), dice=(0.337, 0.129)),
        "SCL-Latent":        dict(psnr=(23.99, 3.42), ssim=(0.662, 0.184), dice=(0.355, 0.106)),
        "CurvatureDynamics": dict(psnr=(22.07, 4.37), ssim=(0.688, 0.155), dice=(0.321, 0.109)),
    },
}


def fmt(mean_std):
    return f"{mean_std[0]:.3f} +/- {mean_std[1]:.3f}"


print("=" * 78)
print("COMPARISON TABLE -- Multi-seed (primary split, 5 seeds)")
print("=" * 78)
rows = []
for name, vals in REFERENCE_RESULTS["multiseed"].items():
    rows.append(dict(model=name, psnr=fmt(vals["psnr"]), ssim=fmt(vals["ssim"]), dice=fmt(vals["dice"])))
if multiseed_summary is not None:
    ms = multiseed_summary.loc["DirectUNetBaseline"]
    rows.append(dict(
        model="DirectUNetBaseline (NEW)",
        psnr=fmt((ms[("psnr_mean", "mean")], ms[("psnr_mean", "std")])),
        ssim=fmt((ms[("ssim_mean", "mean")], ms[("ssim_mean", "std")])),
        dice=fmt((ms[("dice_mean", "mean")], ms[("dice_mean", "std")])),
    ))
print(pd.DataFrame(rows).set_index("model").to_string())

print()
print("=" * 78)
print("COMPARISON TABLE -- k-fold x multi-seed (25 runs)")
print("=" * 78)
rows = []
for name, vals in REFERENCE_RESULTS["kfold"].items():
    rows.append(dict(model=name, psnr=fmt(vals["psnr"]), ssim=fmt(vals["ssim"]), dice=fmt(vals["dice"])))
if kfold_summary_direct is not None:
    ks = kfold_summary_direct.loc["DirectUNetBaseline"]
    rows.append(dict(
        model="DirectUNetBaseline (NEW)",
        psnr=fmt((ks[("psnr_mean", "mean")], ks[("psnr_mean", "std")])),
        ssim=fmt((ks[("ssim_mean", "mean")], ks[("ssim_mean", "std")])),
        dice=fmt((ks[("dice_mean", "mean")], ks[("dice_mean", "std")])),
    ))
print(pd.DataFrame(rows).set_index("model").to_string())

In [ ]:
print()


if kfold_summary_direct is not None:
    direct_dice_mean = kfold_summary_direct.loc["DirectUNetBaseline", ("dice_mean", "mean")]
    direct_dice_std = kfold_summary_direct.loc["DirectUNetBaseline", ("dice_mean", "std")]
    direct_psnr_mean = kfold_summary_direct.loc["DirectUNetBaseline", ("psnr_mean", "mean")]
    direct_ssim_mean = kfold_summary_direct.loc["DirectUNetBaseline", ("ssim_mean", "mean")]

    scl_dice_mean, scl_dice_std = REFERENCE_RESULTS["kfold"]["SCL-Latent"]["dice"]
    base_dice_mean, base_dice_std = REFERENCE_RESULTS["kfold"]["Baseline"]["dice"]
    scl_psnr_mean = REFERENCE_RESULTS["kfold"]["SCL-Latent"]["psnr"][0]
    scl_ssim_mean = REFERENCE_RESULTS["kfold"]["SCL-Latent"]["ssim"][0]

    def direction(new, ref, metric_name, higher_is_better=True):
        diff = new - ref
        if abs(diff) < 1e-9:
            return f"{metric_name}: tied with SCL-Latent ({new:.3f} vs {ref:.3f})."
        better = (diff > 0) if higher_is_better else (diff < 0)
        verdict = "HIGHER (better)" if better else "LOWER (worse)"
        return f"{metric_name}: DirectUNetBaseline = {new:.3f} vs SCL-Latent = {ref:.3f}  -->  {verdict} by {abs(diff):.3f}"

    print(direction(direct_dice_mean, scl_dice_mean, "Dice mean"))
    print(direction(direct_psnr_mean, scl_psnr_mean, "PSNR mean"))
    print(direction(direct_ssim_mean, scl_ssim_mean, "SSIM mean"))
    print()
    print(direction(direct_dice_std, scl_dice_std, "Dice std (LOWER is more stable)", higher_is_better=False))

    print()
